In [1]:
from biopandas.mol2 import PandasMol2
from biopandas.pdb import PandasPdb

import pandas as pd
import numpy as np
import regex as re
import matplotlib.pyplot as plt
import re

import stellargraph as sg
from stellargraph.mapper import PaddedGraphGenerator
from stellargraph.layer import GCNSupervisedGraphClassification
from stellargraph.layer import DeepGraphCNN
from stellargraph import StellarGraph

from sklearn import model_selection
from IPython.display import display, HTML

import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam, RMSprop, SGD
from tensorflow.keras.layers import Dense, Concatenate, Conv1D, Dropout, Flatten, MaxPool1D, concatenate
from tensorflow.keras.losses import binary_crossentropy
from tensorflow.keras.callbacks import EarlyStopping

from sklearn.datasets import make_regression
from sklearn.preprocessing import StandardScaler
from keras.layers import Dense
from keras.models import Sequential
#from keras.optimizers import SGD

import matplotlib.pyplot as plt
import math 

pd.set_option('mode.chained_assignment', None)


2022-03-07 17:44:35.128212: I tensorflow/stream_executor/cuda/cuda_gpu_executor.cc:936] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero
2022-03-07 17:44:35.134306: I tensorflow/stream_executor/cuda/cuda_gpu_executor.cc:936] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero
2022-03-07 17:44:35.134440: I tensorflow/stream_executor/cuda/cuda_gpu_executor.cc:936] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero
2022-03-07 17:44:35.135023: I tensorflow/core/platform/cpu_feature_guard.cc:151] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  AVX2 FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags

In [2]:

import stellargraph as sg

In [3]:
import pandas as pd
import numpy as np

from sklearn import model_selection
from IPython.display import display, HTML

from tensorflow.keras import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.layers import Dense, Conv1D, MaxPool1D, Dropout, Flatten
from tensorflow.keras.losses import binary_crossentropy
import tensorflow as tf



In [4]:
import os
directory="HMDH-validation"
#joined_path = os.path.join("./", directory, a_file)

In [5]:

import glob

#idsA = glob.glob("./arb1/*.mol2")
idsA = glob.glob(os.path.join("./",directory, '*.mol2'))
#idsA = [id.replace("./arb1/", "") for id in idsA]
idsA = [id.replace(("./"+directory+"/"),"") for id in idsA]
idsA = [id.replace(".mol2", "") for id in idsA]

idsA[:5]

['decoys3091', 'active63', 'decoys3063', 'decoys2080', 'active46']

In [6]:
def bond_parser(filename):
    with open(filename, 'r') as f:
        f_text = f.read()
    bonds =  np.array(re.sub(r'\s+', ' ', re.search(r'@<TRIPOS>BOND([a-z0-9\s]*)', f_text).group(1)).split()).reshape((-1, 4))
    
    df_bonds = pd.DataFrame(bonds, columns=['bond_id', 'source', 'target', 'bond_type'])
    #print(df_bonds)
    df_bonds.set_index(['bond_id'], inplace=True)
    #print(df_bonds)
    return df_bonds

def calc_dist(df, node1, node2) : 
    x1, y1, z1 = df.iloc[np.where(df.index == str(node1))][['x', 'y', 'z']].values.flatten()
    x2, y2, z2 = df.iloc[np.where(df.index == str(node2))][['x', 'y', 'z']].values.flatten()

    dist = np.sqrt((x2-x1)**2 + (y2-y1)**2 + (z2-z1)**2) 
    return(dist)   


def CalcSASAFile(filename) : 
  lines = open(filename, 'r')

  tmp_file = open("tmp.txt", "w")

  for line in lines : 
    if re.match("^ATOM", line) : 
      tmp_file.write(line)
    
  tmp_file.close()

  dd = pd.read_csv('tmp.txt', header=None, sep='\n')
  dd = dd[0].str.split('\s+', expand=True)
  dd = dd.iloc[:, -8:-3]
  dd.columns = ['x', 'y', 'z', 'radius', 'area']
  dd.to_csv("parsed_sasa.csv")
  
  cols = dd.columns
  dd[cols] = dd[cols].apply(pd.to_numeric) #, errors='coerce')
  
  return (dd) 



def Merge_Mol2_SASA (mol2, sasa) : 
  N = 1000
  mol2.x = np.round(mol2.x*N).astype(int)
  mol2.y = np.round(mol2.y*N).astype(int)
  mol2.z = np.round(mol2.z*N).astype(int)

  sasa.x = np.round(sasa.x*N).astype(int)
  sasa.y = np.round(sasa.y*N).astype(int)
  sasa.z = np.round(sasa.z*N).astype(int)

  mol2 = mol2.reset_index().merge(sasa, how='left').set_index('atom_id')

  mol2.x = mol2.x / N
  mol2.y = mol2.y / N
  mol2.z = mol2.z / N

  return (mol2)

In [7]:
def bond_parser(filename):
    with open(filename, 'r') as f:
        f_text = f.read()
    bonds =  np.array(re.sub(r'\s+', ' ', re.search(r'@<TRIPOS>BOND([a-z0-9\s]*)', f_text).group(1)).split()).reshape((-1, 4))
    
    df_bonds = pd.DataFrame(bonds, columns=['bond_id', 'source', 'target', 'bond_type'])
    #print(df_bonds)
    df_bonds.set_index(['bond_id'], inplace=True)
    #print(df_bonds)
    return df_bonds

def calc_dist(df, node1, node2) : 
    x1, y1, z1 = df.iloc[np.where(df.index == str(node1))][['x', 'y', 'z']].values.flatten()
    x2, y2, z2 = df.iloc[np.where(df.index == str(node2))][['x', 'y', 'z']].values.flatten()

    dist = np.sqrt((x2-x1)**2 + (y2-y1)**2 + (z2-z1)**2) 
    return(dist)   


def CalcSASAFile(filename) : 
  lines = open(filename, 'r')

  tmp_file = open("tmp.txt", "w")

  for line in lines : 
    if re.match("^ATOM", line) : 
      tmp_file.write(line)
    
  tmp_file.close()

  dd = pd.read_csv('tmp.txt', header=None, sep='\n')
  dd = dd[0].str.split('\s+', expand=True)
  dd = dd.iloc[:, -8:-3]
  dd.columns = ['x', 'y', 'z', 'radius', 'area']
  dd.to_csv("parsed_sasa.csv")
  
  cols = dd.columns
  dd[cols] = dd[cols].apply(pd.to_numeric) #, errors='coerce')
  
  return (dd) 



def Merge_Mol2_SASA (mol2, sasa) : 
  N = 1000
  mol2.x = np.round(mol2.x*N).astype(int)
  mol2.y = np.round(mol2.y*N).astype(int)
  mol2.z = np.round(mol2.z*N).astype(int)

  sasa.x = np.round(sasa.x*N).astype(int)
  sasa.y = np.round(sasa.y*N).astype(int)
  sasa.z = np.round(sasa.z*N).astype(int)

  mol2 = mol2.reset_index().merge(sasa, how='left').set_index('atom_id')

  mol2.x = mol2.x / N
  mol2.y = mol2.y / N
  mol2.z = mol2.z / N

  return (mol2)

In [8]:
atom_props = pd.read_csv("./atom_props.csv")
atom_type = pd.read_csv("./Atom_Type.csv")
atom_type.head(2)
atom_props.head(2)
atom_type.head(2)

,atom_type,C.1,C.2,C.3,C.ar,C.cat,H,N.1,N.2,N.3,...,N.ar,N.pl3,O.2,O.3,O.co2,S.2,S.3,S.o,S.o2,Ot
0,Br,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1
1,C.1,1,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [9]:
def ligand_prep(id):
#id="3pce"

    ligand_file = "./"+directory+"/" + id + ".mol2"
    #ligand_file = "../AI-GCN/Ligand/" + id + "_ligand.mol2"


    ligand = PandasMol2().read_mol2(ligand_file)
    ligand_atoms = ligand.df
    ligand_bonds = bond_parser(ligand_file)


    ligand_atoms.atom_id = ligand_atoms.atom_id.astype("str")
    #print(ligand_atoms)

    mat = '^HOH|^ZN|^CO|^K|^MG|^CA|^CU|^NA|^MN|^NI'

    nh_index_l = (ligand_atoms.iloc[:, 5].str.split(".", expand=True)[0] != "H") #& (~ligand_atoms.iloc[:, 7].str.contains(mat, case=False, regex=True))
    nh_nodes_l = ligand_atoms[nh_index_l].atom_id
    #print(nh_nodes_l)

    ligand_atoms = ligand_atoms.iloc[:, [0, 2, 3, 4, 5, 8]]
    ligand_atoms = ligand_atoms[ligand_atoms.atom_id.isin(nh_nodes_l)]
    ligand_atoms = ligand_atoms.set_index("atom_id")

    ligand_bonds = ligand_bonds[ligand_bonds.source.isin(nh_nodes_l)]
    ligand_bonds = ligand_bonds[ligand_bonds.target.isin(nh_nodes_l)]
    #

    ligand_atoms = ligand_atoms.reset_index().merge(atom_type, how='left', left_on='atom_type', right_on="atom_type").set_index('atom_id')
    ligand_atoms['atom_type'] = ligand_atoms.iloc[:, 3].str.split(".", expand=True)[0]
    ligand_atoms = ligand_atoms.reset_index().merge(atom_props, how='left', left_on='atom_type', right_on="Atom").set_index('atom_id')
    ligand_atoms.drop(["atom_type", "Atom"], axis=1, inplace=True)

    ligand_dists = []
    for s,t in zip(ligand_bonds.source, ligand_bonds.target) : 
        ligand_dists.append(calc_dist(ligand_atoms, s, t))
    ligand_bonds['weight'] = ligand_dists


    mean_x_l=(ligand_atoms.x.sum()/ligand_atoms.x.count())  
    sd_x_l= (pow((ligand_atoms.x-mean_x_l),2)).sum()
    sd_x_l=sd_x_l/ligand_atoms.x.count()
    sd_x_l=math.sqrt(sd_x_l)

    mean_y_l=(ligand_atoms.y.sum()/ligand_atoms.y.count())  
    sd_y_l= (pow((ligand_atoms.y-mean_y_l),2)).sum()
    sd_y_l=sd_y_l/ligand_atoms.y.count()
    sd_y_l=math.sqrt(sd_y_l)  
      #ligand_atoms.y = (ligand_atoms.y - mean_y_l) / sd_x_l

    mean_z_l=(ligand_atoms.z.sum()/ligand_atoms.z.count())  
    sd_z_l= (pow((ligand_atoms.z-mean_z_l),2)).sum()
    sd_z_l=sd_z_l/ligand_atoms.z.count()
    sd_z_l=math.sqrt(sd_z_l)

    mean_charge_l=(ligand_atoms.charge.sum()/ligand_atoms.charge.count())  
    sd_charge_l= (pow((ligand_atoms.charge - mean_charge_l),2)).sum()
    sd_charge_l=sd_charge_l/ligand_atoms.charge.count()
    sd_charge_l=math.sqrt(sd_charge_l) 


    ligand_atoms = (ligand_atoms - ligand_atoms.min()) / (ligand_atoms.max() - ligand_atoms.min())


    ligand_atoms = ligand_atoms.replace(np.nan, 0) 


    ligand_bonds.bond_type=ligand_bonds.bond_type.replace('ar',6)
    ligand_bonds.bond_type=ligand_bonds.bond_type.replace('am',7) 
    ligand_bonds.bond_type=ligand_bonds.bond_type.replace('du',0) 
    ligand_bonds.bond_type=ligand_bonds.bond_type.replace('un',0)   
    #print(ligand_atoms)   

    return (sg.StellarGraph(ligand_atoms, edges = ligand_bonds),id)

In [10]:
print(idsA[0])
res=ligand_prep(idsA[0])

print(res[0].info())
print("="*20)
#print(res[1].info())
print("="*20)
print(res[0].node_features())
print(pd.DataFrame(res[0].node_features()).tail())
print("="*20)
#print(pd.DataFrame(res[1].node_features()).tail())
print("="*20)
print(pd.DataFrame(res[0].node_features()).isna().sum())
#print(pd.DataFrame(res[1].node_features()).isna().sum())

decoys3091
StellarGraph: Undirected multigraph
 Nodes: 20, Edges: 21

 Node types:
  default: [20]
    Features: float32 vector, length 36
    Edge types: default-default->default

 Edge types:
    default-default->default: [21]
        Weights: all 0
        Features: float32 vector, length 1
[[0.         0.         0.         0.56557375 0.         0.
  0.         1.         0.         0.         0.         0.
  0.         0.         0.         0.         0.         0.
  0.         0.         0.         0.         0.         0.
  0.         0.         1.         1.         0.5398609  0.
  0.6        0.09756097 0.5531915  0.         0.         0.        ]
 [0.         0.         0.         0.5685544  0.         0.
  0.         1.         0.         0.         0.         0.
  0.         0.         0.         0.         0.         0.
  0.         0.         0.         0.         0.         0.
  0.         0.         1.         1.         0.5398609  0.
  0.6        0.09756097 0.5531915  0

In [11]:
from multiprocessing import Pool
import math
import time

path="./"+directory+"/"+"All.csv"
selected_ids = pd.read_csv(path).iloc[:,0].values
#print(selected_ids)

pid = []    # Ids which were processed
pgs = []    # List of pocket Graphs
lgs = []    # List of Ligand Graphs 
iid = []    # Ignored IDs which gave error

count = 0

for id in selected_ids :  
  try : 
    #print(id)
    lg = ligand_prep(id)
    lgs.append(lg[0])
    pid.append(lg[1])
    #print(lg[0])
    count += 1
    if (count % 250 == 0) : 
      print (count, "-", id)
  except : 
    iid.append(id)
        
import joblib
joblib.dump((pid,lgs), "DUDE_abl1.var")

250 - decoys1828


['DUDE_abl1.var']

In [12]:
print(len(pid))
path="./"+directory+"/"+"All_Activity.txt"
!sed -i 's:-e::g' $path

351


In [13]:
pid, lgs = joblib.load("DUDE_abl1" + ".var")
pl_graphs = []

path1="./"+directory+"/"+"All.csv"
path2="./"+directory+"/"+"All_Activity.txt"
# iterating over the data
for ind in range(len(lgs)) :
  pl_graphs.append(lgs[ind])


selected_ids = pd.read_csv(path1).iloc[:,0].values
print("Number of IDs common between selected file ",len(selected_ids))
#set(selected_ids)

sel_index = [x for x in range(len(pid)) if pid[x] in selected_ids]
pid = [pid[i] for i in sel_index]
lgs = [lgs[i] for i in sel_index]


target = pd.read_csv(path2, sep="\s+", header=None)

#print(target)
res = [target[target[0] == x][2].values.item() for x in pid]
res = pd.DataFrame(res)
#print("Number of IDs common between selected file and in .var file and contacts ",len(sel_index))

Number of IDs common between selected file  351


In [14]:
res[0]

0      1
1      1
2      1
3      1
4      1
      ..
346    1
347    1
348    1
349    1
350    1
Name: 0, Length: 351, dtype: int64

In [15]:
label=(res[0])
label.name='LABEL'
label=label.astype(dtype='category')
label

0      1
1      1
2      1
3      1
4      1
      ..
346    1
347    1
348    1
349    1
350    1
Name: LABEL, Length: 351, dtype: category
Categories (2, int64): [0, 1]

In [16]:
label = pd.get_dummies(label, drop_first=True)
label

,1
0,1
1,1
2,1
3,1
4,1
...,...
346,1
347,1
348,1
349,1


In [17]:
label.values

array([[1],
       [1],
       [1],
       [1],
       [1],
       [1],
       [1],
       [1],
       [1],
       [1],
       [1],
       [1],
       [1],
       [1],
       [1],
       [1],
       [1],
       [1],
       [1],
       [1],
       [1],
       [1],
       [1],
       [1],
       [1],
       [1],
       [1],
       [1],
       [1],
       [1],
       [1],
       [1],
       [1],
       [1],
       [1],
       [1],
       [1],
       [1],
       [1],
       [1],
       [1],
       [1],
       [1],
       [1],
       [1],
       [1],
       [1],
       [1],
       [1],
       [1],
       [1],
       [1],
       [1],
       [1],
       [1],
       [1],
       [1],
       [1],
       [1],
       [1],
       [1],
       [1],
       [1],
       [1],
       [1],
       [1],
       [1],
       [1],
       [1],
       [1],
       [1],
       [1],
       [1],
       [1],
       [1],
       [1],
       [1],
       [1],
       [1],
       [1],
       [1],
       [1],
       [1],
    

In [18]:
summary = pd.DataFrame(
    [(g.number_of_nodes(), g.number_of_edges()) for g in pl_graphs],
    columns=["nodes", "edges"],
)
summary.describe().round(1)

,nodes,edges
count,351.0,351.0
mean,26.2,27.8
std,8.7,9.6
min,11.0,10.0
25%,19.0,20.0
50%,25.0,27.0
75%,33.0,35.0
max,44.0,47.0


In [19]:
train_graphs, test_graphs = model_selection.train_test_split(
    label, train_size=0.7, test_size=None, stratify=label,
)
train_graphs

,1
294,0
178,0
12,1
326,0
313,0
...,...
50,1
107,1
190,0
251,0


In [20]:
gen = PaddedGraphGenerator(graphs=pl_graphs)
#data_gen = generator.flow(np.arange(0, len(pl_graphs)), batch_size=1, 
#                          symmetric_normalization=False, shuffle=False)


train_gen = gen.flow(
    list(train_graphs.index - 1),
    targets=train_graphs.values,
    batch_size=50,
    symmetric_normalization=False,
)

test_gen = gen.flow(
    list(test_graphs.index - 1),
    targets=test_graphs.values,
    batch_size=1,
    symmetric_normalization=False,
)


In [21]:
test_gen.targets

array([[0],
       [1],
       [0],
       [0],
       [0],
       [0],
       [1],
       [0],
       [1],
       [0],
       [1],
       [1],
       [1],
       [1],
       [1],
       [0],
       [1],
       [0],
       [0],
       [0],
       [1],
       [0],
       [0],
       [1],
       [1],
       [0],
       [0],
       [0],
       [0],
       [1],
       [1],
       [0],
       [1],
       [0],
       [1],
       [0],
       [1],
       [1],
       [1],
       [0],
       [1],
       [1],
       [0],
       [0],
       [1],
       [1],
       [1],
       [0],
       [1],
       [1],
       [1],
       [1],
       [1],
       [0],
       [0],
       [0],
       [0],
       [1],
       [0],
       [1],
       [0],
       [0],
       [0],
       [1],
       [0],
       [1],
       [0],
       [1],
       [1],
       [0],
       [0],
       [0],
       [1],
       [1],
       [0],
       [0],
       [0],
       [1],
       [0],
       [0],
       [0],
       [0],
       [1],
    

In [22]:
import pickle
#filename = directory+"_"+"finalized_model.sav"
#pickle.dump(model, open(filename, 'wb'))

In [23]:
# load the model from disk
loaded_model = pickle.load(open("Uniques_finalized_model.sav", 'rb'))

In [24]:
f=loaded_model.predict(test_gen)
final=pd.DataFrame()
final['obs']=test_graphs
final['pred']=f
final['prediction']=f
final.loc[final.pred>=0.5, "prediction"] = 1
final.loc[final.pred<0.5, "prediction"] = 0
final[final.obs>0]

2022-03-07 17:44:50.033288: I tensorflow/stream_executor/cuda/cuda_blas.cc:1774] TensorFloat-32 will be used for the matrix multiplication. This will only be logged once.
2022-03-07 17:44:50.382804: I tensorflow/stream_executor/cuda/cuda_dnn.cc:368] Loaded cuDNN version 8101
2022-03-07 17:44:50.795970: I tensorflow/core/platform/default/subprocess.cc:304] Start cannot spawn child process: No such file or directory
2022-03-07 17:44:50.796490: I tensorflow/core/platform/default/subprocess.cc:304] Start cannot spawn child process: No such file or directory
2022-03-07 17:44:50.796502: W tensorflow/stream_executor/gpu/asm_compiler.cc:80] Couldn't get ptxas version string: INTERNAL: Couldn't invoke ptxas --version
2022-03-07 17:44:50.797136: I tensorflow/core/platform/default/subprocess.cc:304] Start cannot spawn child process: No such file or directory
2022-03-07 17:44:50.797172: W tensorflow/stream_executor/gpu/redzone_allocator.cc:314] INTERNAL: Failed to launch ptxas
Relying on driver to

,obs,pred,prediction
18,1,1.000000,1.0
33,1,1.000000,1.0
51,1,1.000000,1.0
76,1,1.000000,1.0
91,1,1.000000,1.0
141,1,1.000000,1.0
105,1,1.000000,1.0
159,1,1.000000,1.0
135,1,1.000000,1.0
97,1,1.000000,1.0


In [25]:
f=loaded_model.predict(train_gen)
final=pd.DataFrame()
final['obs']=train_graphs
final['pred']=f
final['prediction']=f
final.loc[final.pred>=0.5, "prediction"] = 1
final.loc[final.pred<0.5, "prediction"] = 0
p = final[final.obs==0]

In [26]:
p[p.prediction==1]

,obs,pred,prediction


In [27]:
f=loaded_model.predict(test_gen)
final=pd.DataFrame()
final['obs']=test_graphs
final['pred']=f
final['prediction']=f
final.loc[final.pred>=0.5, "prediction"] = 1
final.loc[final.pred<0.5, "prediction"] = 0
p=final

f=loaded_model.predict(train_gen)
final=pd.DataFrame()
final['obs']=train_graphs
final['pred']=f
final['prediction']=f
final.loc[final.pred>=0.5, "prediction"] = 1
final.loc[final.pred<0.5, "prediction"] = 0
q=final

import pandas as pd
df1=[p,q]
df=pd.concat(df1)
type(df)

df1=pd.read_csv(path1)
D=[df,df1]
data=pd.concat(D,axis=1)
data

data.to_csv("completePrediciton.csv")

## Validation on known Drugs Ends